# 05 - Merge, Clean, Deduplicate, and Explore ArXiv Labels

This notebook harmonizes `applied`, `theory`, and `survey` CSV files into a single labeled dataset.
It performs cleaning, duplicate/conflict handling, exploratory analysis, visualization, and creates
stratified train/validation/test splits for downstream modeling.


In [6]:
import hashlib
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

DATASET_DIR = Path("Dataset")
DATA_DIR = Path("data")
PLOTS_DIR = Path("plots/eda")
SPLITS_DIR = DATA_DIR / "splits"

for out_dir in [DATA_DIR, PLOTS_DIR, SPLITS_DIR]:
    out_dir.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)


In [7]:
def build_harmonized_frame(
    frame: pd.DataFrame,
    column_map: dict,
    source_dataset: str,
    fixed_label: str | None = None,
) -> pd.DataFrame:
    out = pd.DataFrame()
    for target_col, source_col in column_map.items():
        out[target_col] = frame[source_col] if source_col in frame.columns else np.nan
    if fixed_label is not None:
        out["label"] = fixed_label
    out["source_dataset"] = source_dataset
    return out


canonical_columns = [
    "label",
    "title",
    "abstract",
    "arxiv_id",
    "primary_category",
    "categories",
    "published",
    "arxiv_url",
    "topic",
    "list_name",
    "found_in_arxiv_api",
    "source_dataset",
]

applied_raw = pd.read_csv(DATASET_DIR / "applied_papers_1000.csv")
theory_raw = pd.read_csv(DATASET_DIR / "theory_papers_1000.csv")
survey_raw = pd.read_csv(DATASET_DIR / "survey_papers_901.csv")

applied_map = {
    "label": "label",
    "title": "title",
    "abstract": "abstract",
    "arxiv_id": "arxiv_id",
    "primary_category": "primary_category",
    "categories": "categories",
    "published": "published",
    "arxiv_url": "arxiv_url",
    "topic": "topic",
    "list_name": "list_name",
    "found_in_arxiv_api": "found_in_arxiv_api",
}

theory_map = dict(applied_map)

survey_map = {
    "label": "label",
    "title": "arxiv_title",
    "abstract": "arxiv_abstract",
    "arxiv_id": "arxiv_id",
    "primary_category": "primary_category",
    "categories": "categories",
    "published": "published",
    "arxiv_url": "arxiv_url",
    "topic": "topic",
    "list_name": "list_name",
    "found_in_arxiv_api": "found_in_arxiv_api",
}

applied_df = build_harmonized_frame(applied_raw, applied_map, source_dataset="applied")
theory_df = build_harmonized_frame(theory_raw, theory_map, source_dataset="theory")
survey_df = build_harmonized_frame(
    survey_raw,
    survey_map,
    source_dataset="survey",
    fixed_label="SURVEY",
)

merged_df = pd.concat([applied_df, theory_df, survey_df], ignore_index=True)
merged_df = merged_df[canonical_columns]

print("Merged shape:", merged_df.shape)
print("Columns:", list(merged_df.columns))
merged_df.head(3)


Merged shape: (2901, 12)
Columns: ['label', 'title', 'abstract', 'arxiv_id', 'primary_category', 'categories', 'published', 'arxiv_url', 'topic', 'list_name', 'found_in_arxiv_api', 'source_dataset']


,label,title,abstract,arxiv_id,primary_category,categories,published,arxiv_url,topic,list_name,found_in_arxiv_api,source_dataset
0,APPLIED,Towards Generalizable Robotic Manipulation in ...,Vision-Language-Action (VLA) models excel in s...,2603.15620v1,cs.CV,cs.CV cs.RO,2026-03-16T17:59:57Z,http://arxiv.org/abs/2603.15620v1,NaN,NaN,NaN,applied
1,APPLIED,Mixture-of-Depths Attention,Scaling depth is a key driver for large langua...,2603.15619v1,cs.CL,cs.AI cs.CL,2026-03-16T17:59:55Z,http://arxiv.org/abs/2603.15619v1,NaN,NaN,NaN,applied
2,APPLIED,Look Before Acting: Enhancing Vision Foundatio...,Vision-Language-Action (VLA) models have recen...,2603.15618v1,cs.CV,cs.CV,2026-03-16T17:59:54Z,http://arxiv.org/abs/2603.15618v1,NaN,NaN,NaN,applied


In [8]:
def normalize_text(value: object):
    if pd.isna(value):
        return np.nan
    text = re.sub(r"\s+", " ", str(value).replace("\n", " ")).strip()
    if text == "":
        return np.nan
    return text


clean_df = merged_df.copy()

for col in ["label", "title", "abstract", "primary_category", "categories", "topic", "list_name"]:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].apply(normalize_text)

before_drop_missing = len(clean_df)
clean_df = clean_df.dropna(subset=["title", "abstract"]).copy()
dropped_missing = before_drop_missing - len(clean_df)

clean_df["label"] = clean_df["label"].fillna("UNKNOWN")
clean_df["model_text"] = clean_df["title"] + " " + clean_df["abstract"]

clean_df["title_norm"] = clean_df["title"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
clean_df["abstract_norm"] = clean_df["abstract"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
clean_df["text_fingerprint"] = (
    clean_df["title_norm"] + " || " + clean_df["abstract_norm"]
).map(lambda x: hashlib.md5(x.encode("utf-8")).hexdigest())

print(f"Dropped rows with missing title/abstract: {dropped_missing}")
print("Rows after missing-text cleaning:", len(clean_df))


Dropped rows with missing title/abstract: 0
Rows after missing-text cleaning: 2901


In [9]:
non_null_ids = clean_df.dropna(subset=["arxiv_id"]).copy()
label_per_id = non_null_ids.groupby("arxiv_id")["label"].nunique()
conflicting_ids = label_per_id[label_per_id > 1].index

conflicting_rows = clean_df[clean_df["arxiv_id"].isin(conflicting_ids)].copy()
after_conflict_df = clean_df[~clean_df["arxiv_id"].isin(conflicting_ids)].copy()

same_id_mask = after_conflict_df["arxiv_id"].notna() & after_conflict_df.duplicated(subset=["arxiv_id"], keep="first")
same_id_removed = after_conflict_df[same_id_mask].copy()
after_id_df = after_conflict_df.loc[~same_id_mask].copy()

same_text_mask = after_id_df.duplicated(subset=["text_fingerprint"], keep="first")
text_removed = after_id_df[same_text_mask].copy()
cleaned_df = after_id_df.loc[~same_text_mask].copy()

dedup_audit = pd.DataFrame(
    [
        {"metric": "rows_after_missing_text_drop", "value": int(len(clean_df))},
        {"metric": "conflicting_ids_count", "value": int(len(conflicting_ids))},
        {"metric": "rows_removed_conflicting_labels", "value": int(len(conflicting_rows))},
        {"metric": "rows_removed_same_id", "value": int(len(same_id_removed))},
        {"metric": "rows_removed_same_text", "value": int(len(text_removed))},
        {"metric": "rows_after_dedup", "value": int(len(cleaned_df))},
    ]
)

conflicting_rows.to_csv(DATA_DIR / "dedup_conflicting_rows.csv", index=False)
same_id_removed.to_csv(DATA_DIR / "dedup_same_id_removed.csv", index=False)
text_removed.to_csv(DATA_DIR / "dedup_text_removed.csv", index=False)
dedup_audit.to_csv(DATA_DIR / "dedup_audit_summary.csv", index=False)

print(dedup_audit)
print()
print("Label counts after dedup:")
print(cleaned_df["label"].value_counts())


                            metric  value
0     rows_after_missing_text_drop   2901
1            conflicting_ids_count      2
2  rows_removed_conflicting_labels      4
3             rows_removed_same_id     42
4           rows_removed_same_text      0
5                 rows_after_dedup   2855

Label counts after dedup:
label
APPLIED        998
THEORETICAL    998
SURVEY         859
Name: count, dtype: int64


In [10]:
missingness = (
    cleaned_df.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .reset_index(name="missing_fraction")
    .rename(columns={"index": "column"})
)
label_counts = cleaned_df["label"].value_counts().sort_values(ascending=False).rename("count")
primary_counts = (
    cleaned_df["primary_category"]
    .fillna("<missing>")
    .value_counts()
    .head(20)
    .rename("count")
)

label_counts.to_csv(DATA_DIR / "eda_label_counts.csv", header=True)
missingness.to_csv(DATA_DIR / "eda_missingness.csv", index=False)
primary_counts.to_csv(DATA_DIR / "eda_primary_category_counts.csv", header=True)

cleaned_df["title_length"] = cleaned_df["title"].str.split().str.len()
cleaned_df["abstract_length"] = cleaned_df["abstract"].str.split().str.len()
cleaned_df["published_year"] = pd.to_datetime(cleaned_df["published"], errors="coerce").dt.year

fig, ax = plt.subplots(figsize=(7, 4))
label_counts.plot(kind="bar", ax=ax, color=["#2a9d8f", "#e76f51", "#264653", "#a8dadc"])
ax.set_title("Label Distribution")
ax.set_ylabel("Count")
ax.set_xlabel("Label")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "label_distribution.png", dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
for lbl in sorted(cleaned_df["label"].dropna().unique()):
    vals = cleaned_df.loc[cleaned_df["label"] == lbl, "title_length"].dropna()
    ax.hist(vals, bins=30, alpha=0.45, label=lbl)
ax.set_title("Title Length Distribution by Label")
ax.set_xlabel("Title length (tokens)")
ax.set_ylabel("Frequency")
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / "title_length_by_label.png", dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
for lbl in sorted(cleaned_df["label"].dropna().unique()):
    vals = cleaned_df.loc[cleaned_df["label"] == lbl, "abstract_length"].dropna()
    ax.hist(vals, bins=30, alpha=0.45, label=lbl)
ax.set_title("Abstract Length Distribution by Label")
ax.set_xlabel("Abstract length (tokens)")
ax.set_ylabel("Frequency")
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / "abstract_length_by_label.png", dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 4))
primary_counts.sort_values().plot(kind="barh", ax=ax, color="#457b9d")
ax.set_title("Top Primary Categories")
ax.set_xlabel("Count")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "top_primary_categories.png", dpi=160)
plt.close(fig)

yearly = (
    cleaned_df.dropna(subset=["published_year"])
    .groupby(["published_year", "label"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)
if not yearly.empty:
    yearly.index = yearly.index.astype(int)
    fig, ax = plt.subplots(figsize=(9, 4))
    for lbl in yearly.columns:
        ax.plot(yearly.index, yearly[lbl], marker="o", linewidth=1.5, label=lbl)
    ax.set_title("Papers per Year by Label")
    ax.set_xlabel("Published year")
    ax.set_ylabel("Count")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "papers_per_year_by_label.png", dpi=160)
    plt.close(fig)
else:
    print("No valid published_year values found; skipping year trend plot.")

print("EDA artifacts saved in data/ and plots/eda/.")

EDA artifacts saved in data/ and plots/eda/.


In [11]:
TOKEN_PATTERN = re.compile(r"[A-Za-z][A-Za-z0-9_-]+")


def tokenize(text: str) -> list[str]:
    tokens = [t.lower() for t in TOKEN_PATTERN.findall(text)]
    return [t for t in tokens if len(t) > 2]


token_rows = []
for label, group in cleaned_df.groupby("label"):
    counter = Counter()
    for text in group["model_text"].fillna(""):
        counter.update(tokenize(text))
    for token, count in counter.most_common(30):
        token_rows.append({"label": label, "token": token, "count": int(count)})

token_df = pd.DataFrame(token_rows)
token_df.to_csv(DATA_DIR / "eda_top_tokens_per_label.csv", index=False)
token_df.head(20)


,label,token,count
0,APPLIED,and,6357
1,APPLIED,the,5512
2,APPLIED,for,2481
3,APPLIED,that,2200
4,APPLIED,with,1621
5,APPLIED,this,1292
6,APPLIED,models,1050
7,APPLIED,from,992
8,APPLIED,model,850
9,APPLIED,framework,700


In [12]:
train_df, temp_df = train_test_split(
    cleaned_df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=cleaned_df["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"],
)

train_df.to_csv(SPLITS_DIR / "train_papers.csv", index=False)
val_df.to_csv(SPLITS_DIR / "val_papers.csv", index=False)
test_df.to_csv(SPLITS_DIR / "test_papers.csv", index=False)


def id_overlap(left: pd.DataFrame, right: pd.DataFrame) -> int:
    left_ids = set(left["arxiv_id"].dropna().astype(str))
    right_ids = set(right["arxiv_id"].dropna().astype(str))
    return len(left_ids & right_ids)


print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("ID overlap train/val:", id_overlap(train_df, val_df))
print("ID overlap train/test:", id_overlap(train_df, test_df))
print("ID overlap val/test:", id_overlap(val_df, test_df))

for split_name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{split_name} label distribution:")
    print(frame["label"].value_counts(normalize=True).round(4))


Split sizes: 1998 428 429
ID overlap train/val: 0
ID overlap train/test: 0
ID overlap val/test: 0

train label distribution:
label
THEORETICAL    0.3498
APPLIED        0.3493
SURVEY         0.3008
Name: proportion, dtype: float64

val label distribution:
label
APPLIED        0.3505
THEORETICAL    0.3481
SURVEY         0.3014
Name: proportion, dtype: float64

test label distribution:
label
APPLIED        0.3497
THEORETICAL    0.3497
SURVEY         0.3007
Name: proportion, dtype: float64


In [13]:
cleaned_output = DATA_DIR / "combined_papers_cleaned.csv"
cleaned_df.to_csv(cleaned_output, index=False)

print(f"Saved cleaned dataset to {cleaned_output}")
print("Final cleaned shape:", cleaned_df.shape)
print("Final label counts:")
print(cleaned_df["label"].value_counts())


Saved cleaned dataset to data/combined_papers_cleaned.csv
Final cleaned shape: (2855, 19)
Final label counts:
label
APPLIED        998
THEORETICAL    998
SURVEY         859
Name: count, dtype: int64
